# Kalshi BTC Hourly Arbitrage — Backtester (`arb-v3`)

A **fill-realistic** backtester for the strategy that buys the underpriced bracket
at the top of each hour and exits on convergence.

### The cautionary tale this notebook is built to defeat
A near-identical strategy on Polymarket's 5-min BTC binaries showed **+522× paper
returns** that became **−49.5% live**. The entire gap came from assuming
fills at mid-price. This notebook never makes that mistake — it runs three fill
models side-by-side and refuses to trust the realistic one unless the gap to
optimistic/pessimistic is bounded.

### Design
Each layer lives in its own section. The strategy logic is identical across data
sources (local DuckDB / Kalshi API / synthetic GBM-jumps); only the loader changes.

| § | Layer                         | What it does |
|---|-------------------------------|--------------|
| 1 | Data discovery                | Inventory `data/`, print what we found |
| 2 | Core data models              | `HourlyEpisode`, `Book`, `Trade`, `Position` |
| 3 | Mode A: local DuckDB loader   | 2.5 M real Kalshi quotes + 107 k BTC 1-min bars |
| 4 | Mode B: Kalshi API fetcher    | Public historical candlesticks (no auth) |
| 5 | Mode C: synthetic generator   | GBM + compound-Poisson jumps + LOB model |
| 6 | **Fill simulator**            | optimistic / realistic / pessimistic — the most important module |
| 7 | Strategy engine               | Entry scan, exits, half-Kelly sizing |
| 8 | Portfolio                     | Equity curve, trade ledger |
| 9 | Metrics                       | Sharpe, Calmar, drawdown, exit reasons |
| 10| **Tri-model validation**      | The core deliverable — three equity curves, model-risk ratio |
| 11| Regime analysis               | Slice by BTC realized vol |
| 12| Monte Carlo                   | N-seed fan chart on synthetic data |
| 13| End-to-end demo               | One-button run on first N hours of real data |

### References
- *PredictionMarketBench* (Arora & Malpani, 2026) — deterministic orderbook replay; Bollinger+post-only made +1.67%, naive LLM lost −2.77%.
- *Polymarket live-trading failure* (Liu, 2026) — 522× paper, −49.5% live; **THE** cautionary tale.
- *Market Simulation under Adverse Selection* (arXiv 2409.12721) — track adverse vs non-adverse fills separately.


In [ ]:
# ── Imports & global config ──────────────────────────────────────────
from __future__ import annotations
import os, json, math, random, time, warnings, re
from dataclasses import dataclass, field, asdict
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Optional, Tuple, List, Dict, Iterator

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.set_printoptions(suppress=True, linewidth=120)
pd.set_option("display.float_format", "{:,.4f}".format)

# All tunables in one place. In a multi-file project this would be config.yaml.
CONFIG: Dict = {
    # ── paths ──
    "data_dir":          "data",
    "duckdb_path":       "data/research_datamart/research_backtest.duckdb",
    "output_dir":        "output",
    # ── market structure ──
    "bracket_width_dollars": 100.0,   # KXBTCD strikes are $100 apart
    "kalshi_fee_max":        0.07,
    # ── strategy ──
    "entry_window_seconds":  300,     # consider entries up to t=300 s post-open
    "max_hold_seconds":      300,     # time stop
    "entry_threshold":       0.03,    # 3 ¢ minimum edge
    "exit_target":           0.02,    # +2 ¢ → take profit
    "stop_loss":             0.05,    # −5 ¢ → stop out
    "max_position_pct":      0.05,    # cap any single position at 5 % bankroll
    "kelly_fraction":        0.5,     # half-Kelly
    "starting_bankroll":     10_000.0,
    # ── fill simulation ──
    "latency_ms":            200,
    "fill_model":            "realistic",  # optimistic | realistic | pessimistic
    "adverse_selection_prob": 0.60,
    "adverse_cost_cents":    (1, 3),
    "impact_multiplier":     1.5,
    "pessimistic_book_pct":  0.30,
    "pessimistic_extra_cents": 0.02,
    # ── synthetic ──
    "synth_seed":            42,
    "synth_resolution_seconds": 1,
    # ── diagnostics ──
    "model_risk_threshold":  3.0,
    "min_acceptable_sharpe": 1.0,
}

OUTPUT_DIR = Path(CONFIG["output_dir"])
OUTPUT_DIR.mkdir(exist_ok=True)
print("CONFIG loaded. Output dir:", OUTPUT_DIR.resolve())


## § 1  Data Discovery (Step 0)

Before writing any loader, inspect what's actually in `data/`. The schema dictates
the loader, **never the other way around**. We expect the `sami` branch's data:
~1486 minute-bar Kalshi CSVs + a 47 MB DuckDB datamart with real bid/ask quotes
and BTC 1-minute spot.

In [ ]:
def discover_data(data_dir: str = "data") -> Dict:
    d = Path(data_dir)
    if not d.exists():
        print("DATA INVENTORY: no data/ folder — fall back to synthetic mode.")
        return {"exists": False}
    files = [f for f in d.rglob("*") if f.is_file()]
    by_ext = {}
    for f in files:
        by_ext.setdefault(f.suffix, []).append(f)

    inv = {"exists": True, "total_files": len(files),
           "csv": len(by_ext.get(".csv", [])),
           "duckdb": len(by_ext.get(".duckdb", [])),
           "json": len(by_ext.get(".json", [])),
           "parquet": len(by_ext.get(".parquet", []))}

    # Date span from CSV filenames: kalshi-price-history-kxbtcd-26apr0100-minute.csv
    dates = set()
    for f in by_ext.get(".csv", []):
        m = re.search(r"-(\d{2}[a-z]{3}\d{2})\d{2}-", f.name)
        if m:
            dates.add(m.group(1))

    print("=" * 70)
    print("DATA INVENTORY")
    print("=" * 70)
    print(f"  Total files:    {inv['total_files']}")
    print(f"  CSV files:      {inv['csv']:>5}    (Kalshi minute-bar prices)")
    print(f"  DuckDB files:   {inv['duckdb']:>5}    (research datamart)")
    print(f"  JSON files:     {inv['json']:>5}    (datamart report)")
    print(f"  Parquet files:  {inv['parquet']:>5}")
    print(f"  Unique dates:   {len(dates):>5}")
    if by_ext.get(".csv"):
        print(f"  Sample CSV:     {by_ext['.csv'][0].name}")

    duckdb_path = None
    if by_ext.get(".duckdb"):
        duckdb_path = str(by_ext[".duckdb"][0])
        try:
            import duckdb
            con = duckdb.connect(duckdb_path, read_only=True)
            print(f"\n  DuckDB tables/views ({duckdb_path}):")
            tables = con.execute(
                "SELECT table_name FROM information_schema.tables "
                "WHERE table_schema='main' ORDER BY table_name"
            ).fetchall()
            for (tn,) in tables:
                cnt = con.execute(f"SELECT COUNT(*) FROM {tn}").fetchone()[0]
                print(f"    - {tn:32s} {cnt:>12,} rows")
            try:
                btc = con.execute("SELECT MIN(bucket_start), MAX(bucket_start) FROM btc_1m").fetchone()
                print(f"\n  BTC spot range:  {btc[0]}  →  {btc[1]}")
            except Exception:
                pass
            con.close()
            inv["duckdb_path"] = duckdb_path
        except ImportError:
            print("  (duckdb not installed; install with `pip install duckdb`)")
        except Exception as e:
            print(f"  duckdb inspection failed: {e}")

    print("\n  Verdict: this is real Kalshi quote data + real BTC spot.")
    print("           No need to fully synthesize the LOB — the duckdb has it.")
    print("           Synthetic mode is reserved for stress / Monte Carlo only.")
    return inv

DATA_INVENTORY = discover_data(CONFIG["data_dir"])


## § 2  Core Data Models

A single internal representation keeps the strategy code decoupled from the data
source. Key dataclasses:

- **`Book`** — one snapshot of a tradable bracket: bid/ask, depth, derived true-prob.
- **`HourlyEpisode`** — everything for one hourly cycle: BTC spot path, the bracket
  we'll trade, a minute-indexed `books` DataFrame, and the settlement (1 / 0).
- **`Trade`**, **`Position`**, **`TradeRecord`** — execution + audit objects.

### Bracket construction on the *or-above* universe
Kalshi BTC hourly markets are cumulative ("at or above strike $X"). To trade a
single $100-wide bracket `[X, X+100]`, we replicate it as a two-leg spread:

```
buy   YES_at_or_above[X]    @ ask
sell  YES_at_or_above[X+100] @ bid     ⇔   buy NO_at_or_above[X+100] @ no_ask
─────────────────────────────────────────────
net cost  =  ask_low + (1 − bid_high)
payoff    =  $1 if BTC ∈ [X, X+100] at expiry, else $0
```

This combined-instrument view is what `Book.bid` / `Book.ask` represent.

In [ ]:
@dataclass
class Book:
    timestamp: datetime
    bracket_low: float
    bracket_high: float
    bid: float
    ask: float
    spread: float
    depth: int
    true_prob: Optional[float] = None
    spot: Optional[float] = None
    seconds_into_hour: Optional[int] = None

    @property
    def mid(self) -> float:
        return (self.bid + self.ask) / 2

    @property
    def bracket_center(self) -> float:
        return (self.bracket_low + self.bracket_high) / 2


@dataclass
class HourlyEpisode:
    hour_start: datetime
    hour_end: datetime
    spot_prices: pd.Series
    bracket: Tuple[float, float]
    books: pd.DataFrame
    settlement: Optional[float] = None
    realized_vol: Optional[float] = None
    event_ticker: Optional[str] = None
    strike_quotes: Optional[pd.DataFrame] = None
    settlement_price: Optional[float] = None

    def book_at(self, t: datetime) -> Optional[Book]:
        if self.books.empty:
            return None
        ix = self.books.index.searchsorted(t, side="right") - 1
        if ix < 0:
            return None
        ts = self.books.index[ix]
        row = self.books.iloc[ix]
        spot_now = (float(self.spot_prices.asof(t))
                    if not self.spot_prices.empty else None)
        if spot_now is not None and not np.isfinite(spot_now):
            spot_now = None
        return Book(
            timestamp=ts.to_pydatetime() if hasattr(ts, "to_pydatetime") else ts,
            bracket_low=self.bracket[0], bracket_high=self.bracket[1],
            bid=float(row["bid"]), ask=float(row["ask"]),
            spread=float(row["ask"] - row["bid"]),
            depth=int(row["depth"]),
            spot=spot_now,
            seconds_into_hour=int((t - self.hour_start).total_seconds()),
        )


@dataclass
class Trade:
    timestamp: datetime
    side: str
    qty: int
    fill_price: float
    quoted_price: float
    fees: float
    slippage: float
    adverse: bool = False
    note: str = ""


@dataclass
class Position:
    bracket: Tuple[float, float]
    qty: int
    entry_price: float
    entry_time: datetime
    entry_book: Book


def kalshi_fee(price_dollars: float) -> float:
    p = max(0.0, min(1.0, price_dollars))
    return min(0.07, 0.07 * p / 0.50)

## § 3  Mode A — Local DuckDB Loader

Real bid/ask candles + BTC 1-minute spot. The loader picks **one** bracket per
hour: the `[floor, floor+$100)` containing BTC spot at hour open. This is the
bracket the strategy will try to trade.

`kalshi_quotes` holds 2.5 M observed bid/ask candles tagged
`historical_candle_bidask` — these are real, not interpolated. We forward-fill
to a 1-minute grid for resampling, but never *backward*-fill (that would leak
future quotes into past decisions).

In [ ]:
# KalshiLocalLoader: real Kalshi quotes from data/research_datamart/research_backtest.duckdb
class KalshiLocalLoader:

    def __init__(self, duckdb_path: str, bracket_width: float = 100.0):
        import duckdb
        self.path = duckdb_path
        self.con = duckdb.connect(duckdb_path, read_only=True)
        self.bracket_width = bracket_width

    def list_events(self, limit: Optional[int] = None,
                    only_with_btc: bool = True) -> List[str]:
        q = ("SELECT DISTINCT event_ticker, MIN(open_time) AS ot "
             "FROM kalshi_markets "
             "WHERE is_hourly_kxbtcd = TRUE")
        if only_with_btc:
            q += (" AND open_time >= (SELECT MIN(bucket_start) FROM btc_1m) "
                  "AND open_time <= (SELECT MAX(bucket_start) FROM btc_1m)")
        q += " GROUP BY event_ticker ORDER BY ot"
        if limit:
            q += f" LIMIT {int(limit)}"
        return [r[0] for r in self.con.execute(q).fetchall()]

    def _to_utc(self, dt_val):
        """Convert a DuckDB timestamp (possibly with non-UTC tz) to UTC."""
        ts = pd.Timestamp(dt_val)
        if ts.tzinfo is not None:
            return ts.tz_convert("UTC")
        return ts.tz_localize("UTC")

    def _iso(self, dt_val):
        """Convert any datetime to a UTC ISO string safe for DuckDB params."""
        return self._to_utc(dt_val).isoformat()

    def load_btc_spot(self, t0, t1) -> pd.Series:
        df = self.con.execute(
            "SELECT bucket_start AS ts, close FROM btc_1m "
            "WHERE bucket_start >= ? AND bucket_start <= ? ORDER BY bucket_start",
            [self._iso(t0), self._iso(t1)],
        ).fetchdf()
        if df.empty:
            return pd.Series(dtype=float)
        df["ts"] = pd.to_datetime(df["ts"], utc=True)
        return df.set_index("ts")["close"].astype(float)

    def _load_bracket_books(self, event_ticker, hour_start, hour_end, spot_at_start):
        """Load bracket-level books (two-leg spread) for backward compat."""
        bw = self.bracket_width
        bracket_low = math.floor(float(spot_at_start) / bw) * bw
        bracket_high = bracket_low + bw

        legs = self.con.execute(
            "SELECT market_ticker, floor_strike FROM kalshi_markets "
            "WHERE event_ticker = ? AND floor_strike IN (?, ?)",
            [event_ticker, bracket_low, bracket_high],
        ).fetchdf()
        if len(legs) < 2:
            return None, (bracket_low, bracket_high)
        try:
            t_low  = legs.loc[legs["floor_strike"] == bracket_low,  "market_ticker"].iloc[0]
            t_high = legs.loc[legs["floor_strike"] == bracket_high, "market_ticker"].iloc[0]
        except IndexError:
            return None, (bracket_low, bracket_high)

        hs_s = self._iso(hour_start)
        he_s = self._iso(hour_end)

        def _quotes(ticker):
            df = self.con.execute(
                "SELECT available_at AS ts, yes_bid_close, yes_ask_close "
                "FROM kalshi_quotes WHERE market_ticker = ? "
                "AND available_at BETWEEN ? AND ? ORDER BY available_at",
                [ticker, hs_s, he_s],
            ).fetchdf()
            if df.empty:
                return df
            df["ts"] = pd.to_datetime(df["ts"], utc=True)
            return df.set_index("ts")

        ql = _quotes(t_low)
        qh = _quotes(t_high)
        if ql.empty or qh.empty:
            return None, (bracket_low, bracket_high)

        idx = pd.date_range(hour_start, hour_end, freq="1min", tz="UTC")
        ql_r = ql[["yes_bid_close", "yes_ask_close"]].reindex(idx, method="ffill")
        qh_r = qh[["yes_bid_close", "yes_ask_close"]].reindex(idx, method="ffill")

        books = pd.DataFrame(index=idx)
        books["ask"] = ql_r["yes_ask_close"] - qh_r["yes_bid_close"]
        books["bid"] = ql_r["yes_bid_close"] - qh_r["yes_ask_close"]
        books = books.dropna(subset=["bid", "ask"])
        if books.empty:
            return None, (bracket_low, bracket_high)
        books["bid"] = books["bid"].clip(0.01, 0.99)
        books["ask"] = books["ask"].clip(0.01, 0.99)
        books["ask"] = books[["ask", "bid"]].max(axis=1)
        books["spread"] = (books["ask"] - books["bid"]).clip(lower=0.0)
        books["depth"] = (50.0 + 200.0 / (1.0 + books["spread"] * 5.0)).clip(lower=5).astype(int)
        return books, (bracket_low, bracket_high)

    def load_event(self, event_ticker: str) -> Optional[HourlyEpisode]:
        meta = self.con.execute(
            "SELECT MIN(open_time), MIN(close_time) FROM kalshi_markets "
            "WHERE event_ticker = ?", [event_ticker]
        ).fetchone()
        if not meta or meta[0] is None:
            return None
        hour_start = self._to_utc(meta[0])
        hour_end   = self._to_utc(meta[1])

        spot = self.load_btc_spot(
            hour_start - pd.Timedelta(minutes=65),
            hour_end   + pd.Timedelta(minutes=2),
        )
        if spot.empty:
            return None
        spot_at_start = spot.asof(hour_start)
        if pd.isna(spot_at_start):
            return None

        books, bracket = self._load_bracket_books(
            event_ticker, hour_start, hour_end, spot_at_start)
        if books is None or books.empty:
            return None

        spot_at_end = spot.asof(hour_end)
        sett_price = float(spot_at_end) if pd.notna(spot_at_end) else None
        settlement = None
        if sett_price is not None:
            settlement = 1.0 if (bracket[0] <= sett_price < bracket[1]) else 0.0

        log_ret = np.diff(np.log(spot.values).astype(float))
        rv = float(np.std(log_ret) * math.sqrt(60)) if len(log_ret) >= 5 else 0.0
        if not np.isfinite(rv):
            rv = 0.0

        return HourlyEpisode(
            hour_start=hour_start.to_pydatetime(),
            hour_end=hour_end.to_pydatetime(),
            spot_prices=spot, bracket=bracket, books=books,
            settlement=settlement, realized_vol=rv,
            event_ticker=event_ticker, settlement_price=sett_price,
        )

    def load_event_full(self, event_ticker: str) -> Optional[HourlyEpisode]:
        """Load episode with per-strike quotes for multi-strategy backtesting."""
        ep = self.load_event(event_ticker)
        if ep is None:
            return None

        hs_s = self._iso(ep.hour_start)
        he_s = self._iso(ep.hour_end)

        sq = self.con.execute(
            "SELECT q.available_at AS timestamp, m.floor_strike AS strike, "
            "m.market_ticker AS ticker, q.yes_bid_close AS yes_bid, "
            "q.yes_ask_close AS yes_ask, q.no_ask_exe AS no_ask "
            "FROM kalshi_quotes q "
            "JOIN kalshi_markets m ON q.market_ticker = m.market_ticker "
            "WHERE q.event_ticker = ? "
            "AND q.available_at BETWEEN ? AND ? "
            "ORDER BY q.available_at, m.floor_strike",
            [event_ticker, hs_s, he_s],
        ).fetchdf()

        if not sq.empty:
            sq["timestamp"] = pd.to_datetime(sq["timestamp"], utc=True)
            ep.strike_quotes = sq
        return ep

    def iter_episodes(self, limit: Optional[int] = None) -> Iterator[HourlyEpisode]:
        for et in self.list_events(limit=limit):
            ep = self.load_event(et)
            if ep is not None and not ep.books.empty:
                yield ep

    def iter_episodes_full(self, limit: Optional[int] = None) -> Iterator[HourlyEpisode]:
        for et in self.list_events(limit=limit):
            ep = self.load_event_full(et)
            if ep is not None and not ep.books.empty:
                yield ep

    def close(self):
        self.con.close()

## § 4  Mode B — Kalshi API Fetcher

For supplementing local data when there are gaps. The public historical
candlesticks endpoint requires no auth (~30 req/sec). Tickers follow
`KXBTCD-{YYMMM}{DD}{HH}-T{strike}`.

Not exercised by default — kept here so a one-line swap (`loader = api.fetch(...)`)
makes the pipeline run on freshly-pulled data.

In [ ]:
class KalshiAPIFetcher:
    BASE = "https://api.elections.kalshi.com/trade-api/v2"

    def __init__(self, rate_limit_per_sec: int = 25):
        import requests
        self.s = requests.Session()
        self._min_interval = 1.0 / max(1, rate_limit_per_sec)
        self._last = 0.0

    def _throttle(self):
        wait = self._min_interval - (time.time() - self._last)
        if wait > 0:
            time.sleep(wait)
        self._last = time.time()

    def list_events(self, series_ticker: str = "KXBTC", limit: int = 200) -> List[Dict]:
        self._throttle()
        r = self.s.get(f"{self.BASE}/events",
                       params={"series_ticker": series_ticker, "limit": limit},
                       timeout=10)
        r.raise_for_status()
        return r.json().get("events", [])

    def list_markets_for_event(self, event_ticker: str) -> List[Dict]:
        self._throttle()
        r = self.s.get(f"{self.BASE}/markets",
                       params={"event_ticker": event_ticker, "limit": 1000},
                       timeout=10)
        r.raise_for_status()
        return r.json().get("markets", [])

    def fetch_candlesticks(self, market_ticker: str,
                           t0: datetime, t1: datetime,
                           period_interval: int = 1) -> pd.DataFrame:
        self._throttle()
        r = self.s.get(
            f"{self.BASE}/historical/markets/{market_ticker}/candlesticks",
            params={"start_ts": int(t0.timestamp()),
                    "end_ts":   int(t1.timestamp()),
                    "period_interval": period_interval},
            timeout=10,
        )
        r.raise_for_status()
        candles = r.json().get("candlesticks", [])
        if not candles:
            return pd.DataFrame()
        rows = []
        for c in candles:
            yb = c.get("yes_bid", {}) or {}
            ya = c.get("yes_ask", {}) or {}
            rows.append({
                "ts": pd.Timestamp(c["end_period_ts"], unit="s", tz="UTC"),
                "yes_bid": float(yb.get("close") or 0) / 100.0,
                "yes_ask": float(ya.get("close") or 0) / 100.0,
                "volume": int(c.get("volume", 0) or 0),
                "open_interest": int(c.get("open_interest", 0) or 0),
            })
        return pd.DataFrame(rows).set_index("ts")

print("KalshiAPIFetcher ready (no API call made — invoke .fetch_candlesticks for live pulls).")


## § 5  Mode C — Synthetic Generator

For Monte Carlo and stress tests. Two pieces:

1. **Spot path** — GBM + compound-Poisson jumps, 1-second resolution. Calibrate
   `sigma_annual` from realized vol of the local data.
2. **Synthetic LOB** — bid/ask/spread/depth as functions of (a) seconds-into-hour
   (liquidity fills exponentially over ~120 s), (b) distance from spot to bracket
   center (logistic true-prob), (c) realized vol.

Defaults: 0.08 base spread, 40 s liquidity time-constant, max depth 195. These
are reasonable priors for KXBTCD hourly markets and can be re-calibrated against
real data once we have enough episodes.

In [ ]:
def gbm_jumps_spot(s0: float, n_seconds: int, sigma_annual: float,
                   jump_intensity_per_hour: float = 4.0,
                   jump_size_pct: float = 0.002,
                   seed: Optional[int] = None) -> np.ndarray:
    rng = np.random.default_rng(seed)
    dt = 1.0 / (252 * 24 * 3600)              # 1 second in years
    z = rng.standard_normal(n_seconds)
    log_ret = -0.5 * sigma_annual ** 2 * dt + sigma_annual * math.sqrt(dt) * z
    p_jump = jump_intensity_per_hour / 3600.0
    jumps = (rng.binomial(1, p_jump, n_seconds)
             * rng.choice([-1, 1], n_seconds)
             * jump_size_pct)
    log_ret += jumps
    return s0 * np.exp(np.cumsum(log_ret))


def estimate_sigma_annual(spot: pd.Series) -> float:
    if len(spot) < 30:
        return 0.65
    log_ret = np.log(spot.astype(float)).diff().dropna()
    return float(log_ret.std() * math.sqrt(252 * 24 * 60))


def synthetic_book(spot: float, bracket: Tuple[float, float],
                   seconds_into_hour: int, realized_vol: float,
                   rng: Optional[np.random.Generator] = None) -> Dict[str, float]:
    if rng is None:
        rng = np.random.default_rng()
    bl, bh = bracket
    bw = max(1.0, bh - bl)
    bc = (bl + bh) / 2
    dist = abs(spot - bc) / bw
    true_prob = 1.0 / (1.0 + math.exp(4.0 * (dist - 0.5)))
    true_prob = float(np.clip(true_prob, 0.01, 0.99))
    liq = 1.0 - math.exp(-seconds_into_hour / 40.0)
    spread = 0.08 * (1.0 - liq * 0.85) * (1.0 + realized_vol * 2.0)
    depth = int(5 + 195 * liq + rng.uniform(0, 30))
    bid = max(0.01, true_prob - spread / 2)
    ask = min(0.99, true_prob + spread / 2)
    return {"bid": bid, "ask": ask, "spread": ask - bid,
            "depth": depth, "true_prob": true_prob}


def synthetic_episode(hour_start: datetime, s0: float = 70_000.0,
                      sigma_annual: float = 0.65,
                      bracket_width: float = 100.0,
                      seed: Optional[int] = None) -> HourlyEpisode:
    rng = np.random.default_rng(seed)
    n_sec = 3600
    spot_arr = gbm_jumps_spot(s0, n_sec, sigma_annual, seed=seed)
    spot_idx = pd.date_range(hour_start, periods=n_sec, freq="1s", tz="UTC")
    spot = pd.Series(spot_arr, index=spot_idx)

    bracket_low = math.floor(s0 / bracket_width) * bracket_width
    bracket_high = bracket_low + bracket_width

    minutes = pd.date_range(hour_start, hour_start + pd.Timedelta(minutes=59),
                            freq="1min", tz="UTC")
    rows = []
    for t in minutes:
        sec_in = int((t - hour_start).total_seconds())
        s_t = float(spot.asof(t))
        b = synthetic_book(s_t, (bracket_low, bracket_high), sec_in,
                           realized_vol=sigma_annual / math.sqrt(252 * 24),
                           rng=rng)
        rows.append({"ts": t, **b})
    books = pd.DataFrame(rows).set_index("ts")[["bid", "ask", "spread", "depth"]]

    spot_end = float(spot.iloc[-1])
    settlement = 1.0 if (bracket_low <= spot_end < bracket_high) else 0.0
    log_ret = np.diff(np.log(spot_arr))
    rv = float(np.std(log_ret) * math.sqrt(60)) if len(log_ret) > 5 else 0.0

    return HourlyEpisode(
        hour_start=hour_start,
        hour_end=hour_start + pd.Timedelta(hours=1),
        spot_prices=spot,
        bracket=(bracket_low, bracket_high),
        books=books, settlement=settlement, realized_vol=rv,
        event_ticker=f"SYNTH-{hour_start:%Y%m%d-%H}",
    )

print("Synthetic generator ready. Quick sanity check…")
_ep = synthetic_episode(datetime(2026, 4, 1, 0, 0, tzinfo=timezone.utc), seed=0)
print(f"  bracket={_ep.bracket}  settlement={_ep.settlement}  rv={_ep.realized_vol:.5f}")
print(f"  first book: bid={_ep.books['bid'].iloc[0]:.3f}  ask={_ep.books['ask'].iloc[0]:.3f}  depth={_ep.books['depth'].iloc[0]}")
print(f"  last  book: bid={_ep.books['bid'].iloc[-1]:.3f}  ask={_ep.books['ask'].iloc[-1]:.3f}  depth={_ep.books['depth'].iloc[-1]}")


## § 6  Fill Simulator — the core anti-fragility module

This is where backtests lie. Three models, identical interface:

### Model 1 — Optimistic *(wrong on purpose)*
- Fill at quoted bid/ask. Full size. No impact, no adverse selection.
- This is the model that produced **+522×** paper performance on Polymarket.
- Included here only to **measure** how wrong it is.

### Model 2 — Realistic *(the one we trust)*
- **Market impact** = `(qty / depth) × spread × 1.5` shifts the fill price against you.
- **Adverse selection** = 60 % chance an informed counterparty crossed you,
  adding 1–3 ¢ of cost.
- **Partial fills** = `P(full fill) = min(1, depth / (1.5 × qty))`.
- **Latency** is modelled in the strategy layer (we observe at `t`, act at `t + lag`).

### Model 3 — Pessimistic *(worst-case)*
- Fill at `ask + 2 ¢` (buys) or `bid − 2 ¢` (sells).
- Only 30 % of quoted depth is reachable.
- Always treat as adverse.

The **gap between Optimistic and Pessimistic** measures model risk. A strategy
whose realistic PnL is dwarfed by that gap is fragile and not deployable.

In [ ]:
@dataclass
class FillResult:
    fill_price: float
    qty: int
    fees: float
    slippage: float           # signed; >0 = paid more than quoted
    adverse: bool
    note: str = ""


class FillSimulator:
    def __init__(self, model: str = "realistic", config: Dict = None,
                 seed: Optional[int] = None):
        if model not in ("optimistic", "realistic", "pessimistic"):
            raise ValueError(f"unknown fill model: {model!r}")
        self.model = model
        self.cfg = config or CONFIG
        s = seed if seed is not None else self.cfg.get("synth_seed", 42)
        self.rng = np.random.default_rng(s)

    # ── BUY ───────────────────────────────────────────────────────────
    def fill_buy(self, book: Book, requested_qty: int) -> FillResult:
        if requested_qty <= 0:
            return FillResult(0.0, 0, 0.0, 0.0, False, "no qty")
        if self.model == "optimistic":
            qty = requested_qty
            fp = book.ask
            return FillResult(fp, qty, kalshi_fee(fp) * qty, 0.0, False, "opt")
        if self.model == "pessimistic":
            avail = max(1, int(book.depth * self.cfg["pessimistic_book_pct"]))
            qty = min(requested_qty, avail)
            fp = min(0.99, book.ask + self.cfg["pessimistic_extra_cents"])
            return FillResult(fp, qty, kalshi_fee(fp) * qty,
                              fp - book.ask, True, "pess")
        # realistic
        avail = max(1, book.depth)
        fill_prob = min(1.0, avail / max(1, int(requested_qty * 1.5)))
        if self.rng.random() <= fill_prob:
            qty = requested_qty
        else:
            qty = max(1, int(round(requested_qty * fill_prob)))
        impact = (qty / avail) * book.spread * self.cfg["impact_multiplier"]
        adverse = self.rng.random() < self.cfg["adverse_selection_prob"]
        ad_cost = (self.rng.uniform(*self.cfg["adverse_cost_cents"]) / 100.0
                   if adverse else 0.0)
        fp = min(0.99, book.ask + impact + ad_cost)
        return FillResult(fp, qty, kalshi_fee(fp) * qty,
                          fp - book.ask, adverse,
                          f"impact={impact:.4f} adv={int(adverse)}")

    # ── SELL ──────────────────────────────────────────────────────────
    def fill_sell(self, book: Book, requested_qty: int) -> FillResult:
        if requested_qty <= 0:
            return FillResult(0.0, 0, 0.0, 0.0, False, "no qty")
        if self.model == "optimistic":
            qty = requested_qty
            fp = book.bid
            return FillResult(fp, qty, kalshi_fee(fp) * qty, 0.0, False, "opt")
        if self.model == "pessimistic":
            avail = max(1, int(book.depth * self.cfg["pessimistic_book_pct"]))
            qty = min(requested_qty, avail)
            fp = max(0.01, book.bid - self.cfg["pessimistic_extra_cents"])
            return FillResult(fp, qty, kalshi_fee(fp) * qty,
                              book.bid - fp, True, "pess")
        avail = max(1, book.depth)
        fill_prob = min(1.0, avail / max(1, int(requested_qty * 1.5)))
        if self.rng.random() <= fill_prob:
            qty = requested_qty
        else:
            qty = max(1, int(round(requested_qty * fill_prob)))
        impact = (qty / avail) * book.spread * self.cfg["impact_multiplier"]
        adverse = self.rng.random() < self.cfg["adverse_selection_prob"]
        ad_cost = (self.rng.uniform(*self.cfg["adverse_cost_cents"]) / 100.0
                   if adverse else 0.0)
        fp = max(0.01, book.bid - impact - ad_cost)
        return FillResult(fp, qty, kalshi_fee(fp) * qty,
                          book.bid - fp, adverse,
                          f"impact={impact:.4f} adv={int(adverse)}")

print("FillSimulator ready. Three models live behind the same .fill_buy/.fill_sell interface.")


## § 7  Multi-Strategy Engine

Three independent strategies sharing execution infrastructure. Each exploits a
different structural inefficiency in Kalshi BTC hourly markets.

| # | Strategy | Edge type | Risk |
|---|----------|-----------|------|
| 1 | **YES+NO Parity** | `YES_ask + NO_ask < $1 − fees` | Risk-free (both legs must fill) |
| 2 | **Cross-Strike Monotonicity** | `YES_bid(K_hi) > YES_ask(K_lo)` | Risk-free (locked profit) |
| 3 | **Expiry Convergence** | Deep ITM contract below fair value, last 5 min | Near risk-free (tail risk) |

All trades settle at expiry. Strategies 1–2 have guaranteed minimum payoff;
Strategy 3 uses a lognormal model restricted to >95% fair value where model
error is negligible.

In [ ]:
CONFIG.setdefault("kalshi_fee_cents", 0.7)
CONFIG.setdefault("cross_arb_min_profit_cents", 1.0)
CONFIG.setdefault("convergence_max_secs", 300)
CONFIG.setdefault("convergence_min_fair", 0.95)
CONFIG.setdefault("convergence_min_edge_cents", 3.0)
CONFIG.setdefault("arb_max_dollars_per_trade", 1000)
CONFIG.setdefault("arb_sigma_window_min", 60)
CONFIG.setdefault("max_position_per_strike", 50)


@dataclass
class TradeRecord:
    episode_idx: int
    hour_start: datetime
    bracket: Tuple[float, float]
    side: str = "LONG"
    strategy: str = ""
    qty: int = 0
    entry_time: Optional[datetime] = None
    entry_price: Optional[float] = None
    entry_quoted: Optional[float] = None
    exit_time: Optional[datetime] = None
    exit_price: Optional[float] = None
    exit_quoted: Optional[float] = None
    fees: float = 0.0
    pnl: float = 0.0
    exit_reason: str = "no_entry"
    settlement: Optional[float] = None
    fill_model: str = "realistic"
    spot_at_entry: Optional[float] = None
    spot_at_exit: Optional[float] = None
    edge_at_entry: Optional[float] = None
    realized_vol: Optional[float] = None
    adverse_entry: bool = False
    adverse_exit: bool = False
    strike: Optional[float] = None
    strike2: Optional[float] = None


def _norm_cdf(x: float) -> float:
    return 0.5 * math.erfc(-x / math.sqrt(2))


def _strike_book(yb, ya, ts, spot, depth=50):
    """Wrap individual strike quotes into a Book for FillSimulator."""
    return Book(timestamp=ts, bracket_low=0, bracket_high=0,
                bid=float(yb), ask=float(ya),
                spread=max(0.0, float(ya - yb)), depth=depth, spot=spot)


class HourlyArbStrategy:
    """Multi-strategy engine: parity arb, monotonicity arb, expiry convergence.

    Returns List[TradeRecord] — zero, one, or many trades per episode.
    """

    def __init__(self, fill_sim: FillSimulator, config: Dict = None):
        self.fill = fill_sim
        self.cfg = config or CONFIG

    def _sigma(self, spots, ts, win=60):
        r = spots.loc[:ts].tail(win + 1)
        if len(r) < max(10, win // 2):
            return None
        s = float(np.std(np.diff(np.log(r.values.astype(float)))) * np.sqrt(525960))
        return s if np.isfinite(s) and s > 0 else None

    def _size(self, price, bankroll):
        cap = min(self.cfg.get("arb_max_dollars_per_trade", 1000),
                  bankroll * self.cfg.get("max_position_pct", 0.05))
        return max(1, min(self.cfg.get("max_position_per_strike", 50),
                          int(cap / max(0.01, price))))

    # ── Strategy 1: YES+NO Parity Arbitrage ──────────────────────────

    def _scan_parity(self, snap, ts, spot, ep, bankroll, episode_idx, used):
        records = []
        for _, r in snap.iterrows():
            K = r["strike"]
            if K in used:
                continue
            ya, na = r["yes_ask"], r["no_ask"]
            if pd.isna(ya) or pd.isna(na) or ya <= 0.01 or na <= 0.01:
                continue
            if ya >= 0.99 or na >= 0.99:
                continue
            combined = ya + na
            fees_est = kalshi_fee(ya) + kalshi_fee(na)
            edge = 1.00 - combined - fees_est
            if edge <= 0:
                continue
            qty = self._size(combined, bankroll)
            bk_y = _strike_book(r.get("yes_bid", ya - 0.01), ya, ts, spot)
            bk_n = _strike_book(max(0.01, 1 - ya), na, ts, spot)
            fy = self.fill.fill_buy(bk_y, qty)
            fn = self.fill.fill_buy(bk_n, qty)
            if fy.qty <= 0 or fn.qty <= 0:
                continue
            fq = min(fy.qty, fn.qty)
            cost = fy.fill_price + fn.fill_price
            tf = fy.fees + fn.fees
            pnl = (1.00 - cost) * fq - tf
            used.add(K)
            records.append(TradeRecord(
                episode_idx=episode_idx, hour_start=ep.hour_start,
                bracket=ep.bracket, side="PAIR", strategy="parity",
                qty=fq, entry_time=ts, entry_price=cost, entry_quoted=combined,
                exit_time=ep.hour_end, exit_price=1.0, exit_quoted=1.0,
                fees=tf, pnl=pnl,
                exit_reason="expiry_win" if pnl > 0 else "expiry_loss",
                settlement=1.0, fill_model=self.fill.model,
                spot_at_entry=spot, edge_at_entry=edge,
                realized_vol=ep.realized_vol, strike=K,
                adverse_entry=fy.adverse or fn.adverse))
        return records

    # ── Strategy 2: Cross-Strike Monotonicity Arbitrage ──────────────

    def _scan_monotonicity(self, snap, ts, spot, ep, bankroll, episode_idx, used):
        records = []
        valid = snap.dropna(subset=["yes_bid", "yes_ask"])
        valid = valid[(valid["yes_bid"] > 0.01) & (valid["yes_ask"] < 0.99)]
        if len(valid) < 2:
            return records

        sorted_s = valid.sort_values("strike")
        min_ask = float("inf")
        min_ask_row = None
        sett_btc = ep.settlement_price

        for _, r in sorted_s.iterrows():
            K = r["strike"]
            yb, ya = float(r["yes_bid"]), float(r["yes_ask"])

            if min_ask_row is not None and yb > min_ask:
                K_lo = float(min_ask_row["strike"])
                if K_lo not in used and K not in used:
                    na_hi = float(r["no_ask"]) if not pd.isna(r["no_ask"]) else 1.0 - yb
                    cost_quoted = float(min_ask_row["yes_ask"]) + na_hi
                    exact_fees = kalshi_fee(float(min_ask_row["yes_ask"])) + kalshi_fee(na_hi)
                    profit_est = 1.0 - cost_quoted - exact_fees
                    min_profit = self.cfg.get("cross_arb_min_profit_cents", 1.0) / 100

                    if profit_est > min_profit:
                        qty = self._size(cost_quoted, bankroll)
                        bk_lo = _strike_book(
                            float(min_ask_row["yes_bid"]),
                            float(min_ask_row["yes_ask"]), ts, spot)
                        bk_hi = _strike_book(max(0.01, 1 - ya), na_hi, ts, spot)
                        flo = self.fill.fill_buy(bk_lo, qty)
                        fhi = self.fill.fill_buy(bk_hi, qty)
                        if flo.qty > 0 and fhi.qty > 0:
                            fq = min(flo.qty, fhi.qty)
                            ec = flo.fill_price + fhi.fill_price
                            tf = flo.fees + fhi.fees
                            if sett_btc is not None:
                                if sett_btc >= K:
                                    sv = 1.0
                                elif sett_btc >= K_lo:
                                    sv = 2.0
                                else:
                                    sv = 1.0
                            else:
                                sv = 1.0
                            pnl = (sv - ec) * fq - tf
                            used.add(K_lo)
                            used.add(K)
                            records.append(TradeRecord(
                                episode_idx=episode_idx, hour_start=ep.hour_start,
                                bracket=ep.bracket, side="PAIR",
                                strategy="monotonicity",
                                qty=fq, entry_time=ts, entry_price=ec,
                                entry_quoted=cost_quoted,
                                exit_time=ep.hour_end, exit_price=sv,
                                exit_quoted=sv, fees=tf, pnl=pnl,
                                exit_reason="expiry_win" if pnl > 0 else "expiry_loss",
                                settlement=sv, fill_model=self.fill.model,
                                spot_at_entry=spot,
                                edge_at_entry=yb - min_ask,
                                realized_vol=ep.realized_vol,
                                strike=K_lo, strike2=K,
                                adverse_entry=flo.adverse or fhi.adverse))
                            break

            if ya < min_ask:
                min_ask = ya
                min_ask_row = r

        return records

    # ── Strategy 3: Expiry Convergence ────────────────────────────────

    def _scan_convergence(self, snap, ts, spot, secs, sigma, ep,
                          bankroll, episode_idx, used):
        records = []
        conv_max = self.cfg.get("convergence_max_secs", 300)
        if secs > conv_max or secs < 10 or sigma is None:
            return records

        min_fair = self.cfg.get("convergence_min_fair", 0.95)
        min_edge = self.cfg.get("convergence_min_edge_cents", 3.0) / 100
        sig_s = sigma / math.sqrt(365.25 * 24 * 3600)
        sig_rem = sig_s * spot * math.sqrt(secs)
        if sig_rem <= 0:
            return records
        sett_btc = ep.settlement_price

        for _, r in snap.iterrows():
            K = float(r["strike"])
            if K in used:
                continue
            ya = r["yes_ask"]
            na = r["no_ask"]

            d = abs(spot - K) / sig_rem
            fair = _norm_cdf(d) if spot > K else 1 - _norm_cdf(d)

            # YES side: deep ITM (spot >> K)
            if fair > min_fair and not pd.isna(ya) and 0.01 < ya < 0.99:
                fee = kalshi_fee(ya)
                edge = fair - ya - fee
                if edge > min_edge:
                    qty = self._size(ya, bankroll)
                    bk = _strike_book(
                        r.get("yes_bid", ya - 0.01), ya, ts, spot)
                    fr = self.fill.fill_buy(bk, qty)
                    if fr.qty > 0:
                        sett = 1.0 if (sett_btc is not None
                                       and sett_btc >= K) else 0.0
                        pnl = (sett - fr.fill_price) * fr.qty - fr.fees
                        used.add(K)
                        records.append(TradeRecord(
                            episode_idx=episode_idx,
                            hour_start=ep.hour_start,
                            bracket=ep.bracket, side="YES",
                            strategy="convergence",
                            qty=fr.qty, entry_time=ts,
                            entry_price=fr.fill_price,
                            entry_quoted=ya, exit_time=ep.hour_end,
                            exit_price=sett, exit_quoted=sett,
                            fees=fr.fees, pnl=pnl,
                            exit_reason="expiry_win" if sett > 0.5
                                        else "expiry_loss",
                            settlement=sett, fill_model=self.fill.model,
                            spot_at_entry=spot, edge_at_entry=edge,
                            realized_vol=ep.realized_vol, strike=K,
                            adverse_entry=fr.adverse))
                        continue

            # NO side: deep OTM (spot << K)
            no_fair = 1 - fair
            if no_fair > min_fair and not pd.isna(na) and 0.01 < na < 0.99:
                fee = kalshi_fee(na)
                edge = no_fair - na - fee
                if edge > min_edge:
                    qty = self._size(na, bankroll)
                    bk = _strike_book(
                        max(0.01, 1 - r.get("yes_ask", na)), na, ts, spot)
                    fr = self.fill.fill_buy(bk, qty)
                    if fr.qty > 0:
                        sett = 1.0 if (sett_btc is not None
                                       and sett_btc < K) else 0.0
                        pnl = (sett - fr.fill_price) * fr.qty - fr.fees
                        used.add(K)
                        records.append(TradeRecord(
                            episode_idx=episode_idx,
                            hour_start=ep.hour_start,
                            bracket=ep.bracket, side="NO",
                            strategy="convergence",
                            qty=fr.qty, entry_time=ts,
                            entry_price=fr.fill_price,
                            entry_quoted=na, exit_time=ep.hour_end,
                            exit_price=sett, exit_quoted=sett,
                            fees=fr.fees, pnl=pnl,
                            exit_reason="expiry_win" if sett > 0.5
                                        else "expiry_loss",
                            settlement=sett, fill_model=self.fill.model,
                            spot_at_entry=spot, edge_at_entry=edge,
                            realized_vol=ep.realized_vol, strike=K,
                            adverse_entry=fr.adverse))

        return records

    # ── Coordinator ───────────────────────────────────────────────────

    def run(self, ep: HourlyEpisode, bankroll: float,
            episode_idx: int = 0) -> List[TradeRecord]:
        if ep.strike_quotes is None or ep.strike_quotes.empty:
            return []

        records: List[TradeRecord] = []
        used: set = set()
        sigma_win = self.cfg.get("arb_sigma_window_min", 60)

        for ts, snap in ep.strike_quotes.groupby("timestamp"):
            spot = (float(ep.spot_prices.asof(ts))
                    if not ep.spot_prices.empty else None)
            if spot is None or not np.isfinite(spot):
                continue
            secs = (ep.hour_end - ts).total_seconds()
            if secs <= 0:
                continue

            # Strategy 1
            recs = self._scan_parity(
                snap, ts, spot, ep, bankroll, episode_idx, used)
            for r in recs:
                records.append(r)
                bankroll += r.pnl

            # Strategy 2
            recs = self._scan_monotonicity(
                snap, ts, spot, ep, bankroll, episode_idx, used)
            for r in recs:
                records.append(r)
                bankroll += r.pnl

            # Strategy 3 (last 5 min only)
            sigma = self._sigma(ep.spot_prices, ts, sigma_win)
            recs = self._scan_convergence(
                snap, ts, spot, secs, sigma, ep,
                bankroll, episode_idx, used)
            for r in recs:
                records.append(r)
                bankroll += r.pnl

        return records


print("Multi-strategy engine loaded: parity + monotonicity + convergence")

## § 8  Portfolio

Tracks bankroll, equity curve, and the trade ledger. PnL has already been
charged fees by the strategy.

In [ ]:
class Portfolio:
    def __init__(self, starting_bankroll: float = 10_000.0):
        self.start = float(starting_bankroll)
        self.bankroll = float(starting_bankroll)
        self.equity_curve: List[Tuple[datetime, float]] = [
            (datetime.now(timezone.utc), self.bankroll)
        ]
        self.records: List[TradeRecord] = []

    def apply(self, rec: TradeRecord):
        self.records.append(rec)
        if rec.entry_time is not None:
            self.bankroll += rec.pnl
            t_mark = rec.exit_time or rec.entry_time
            self.equity_curve.append((t_mark, self.bankroll))

    def equity_df(self) -> pd.DataFrame:
        return pd.DataFrame(self.equity_curve, columns=["ts", "equity"]).set_index("ts")

    def trades_df(self) -> pd.DataFrame:
        if not self.records:
            return pd.DataFrame()
        return pd.DataFrame([asdict(r) for r in self.records])


## § 9  Metrics

Per-run summary: PnL, win rate, Sharpe (annualized × √(24·365)), Calmar,
max drawdown, profit factor, slippage, adverse-fill share, fill rate, exit-reason
breakdown.

In [ ]:
def compute_metrics(portfolio: Portfolio, total_episodes: int) -> Dict:
    df = portfolio.trades_df()
    eq = portfolio.equity_df()
    out = {
        "starting_bankroll":   portfolio.start,
        "ending_bankroll":     portfolio.bankroll,
        "total_pnl_dollars":   portfolio.bankroll - portfolio.start,
        "total_pnl_pct":       (portfolio.bankroll / portfolio.start - 1) * 100.0,
        "n_episodes":          total_episodes,
        "n_entries":           int((df.get("entry_time").notna()).sum()) if not df.empty else 0,
        "n_records":           len(df),
    }
    out["fill_rate_pct"] = (out["n_entries"] / max(1, total_episodes)) * 100.0
    if df.empty or out["n_entries"] == 0:
        return out
    entered = df[df["entry_time"].notna()].copy()
    wins = entered[entered["pnl"] > 0]
    out["win_rate_pct"]      = len(wins) / max(1, len(entered)) * 100.0
    out["avg_pnl_per_trade"] = float(entered["pnl"].mean())
    out["median_pnl"]        = float(entered["pnl"].median())
    gross_win  = float(entered.loc[entered["pnl"] > 0, "pnl"].sum())
    gross_loss = float(-entered.loc[entered["pnl"] < 0, "pnl"].sum())
    out["profit_factor"] = (gross_win / gross_loss) if gross_loss > 0 else float("inf")

    if "entry_quoted" in entered.columns:
        slip = (entered["entry_price"] - entered["entry_quoted"]).abs()
        out["avg_entry_slippage_dollars"] = float(slip.mean())
    out["adverse_entry_rate_pct"] = float(entered["adverse_entry"].mean()) * 100.0

    if len(eq) > 2:
        rets = eq["equity"].pct_change().dropna()
        if rets.std() > 0:
            out["sharpe_annualized"] = float(rets.mean() / rets.std() * math.sqrt(24 * 365))
        else:
            out["sharpe_annualized"] = 0.0
        cum_max = eq["equity"].cummax()
        dd = (eq["equity"] - cum_max) / cum_max
        out["max_drawdown_pct"] = float(dd.min() * 100.0)
        if total_episodes > 0:
            ann_return = out["total_pnl_pct"] * (24 * 365 / total_episodes)
            mdd = abs(out["max_drawdown_pct"])
            out["calmar"] = (ann_return / mdd) if mdd > 0 else float("inf")

    out["exit_reasons"] = entered["exit_reason"].value_counts().to_dict()
    return out


def print_metrics(label: str, m: Dict):
    print(f"\n── {label} ──")
    keys = ["total_pnl_dollars", "total_pnl_pct", "n_entries",
            "fill_rate_pct", "win_rate_pct", "avg_pnl_per_trade",
            "profit_factor", "avg_entry_slippage_dollars",
            "adverse_entry_rate_pct", "sharpe_annualized",
            "max_drawdown_pct", "calmar"]
    for k in keys:
        if k in m:
            v = m[k]
            if isinstance(v, float):
                print(f"  {k:32s} {v:>12,.4f}")
            else:
                print(f"  {k:32s} {v}")
    if "exit_reasons" in m:
        print(f"  exit_reasons:                  {m['exit_reasons']}")


## § 10  Tri-Model Validation — *the core deliverable*

Run all three fill models on the *same* episodes and compare. Definitions:

```
model_risk_ratio = |PnL_optimistic − PnL_pessimistic| / max(|PnL_realistic|, 1)
fragile          = model_risk_ratio > 3
```

The realistic model is the **only** one that matters for go/no-go. The other two
calibrate how much you should trust it.

In [ ]:
def run_tri_model_validation(episodes: List[HourlyEpisode],
                             starting_bankroll: float = 10_000.0,
                             config: Dict = None,
                             verbose: bool = False) -> Dict:
    cfg = config or CONFIG
    out: Dict[str, Dict] = {}
    for model in ("optimistic", "realistic", "pessimistic"):
        pf = Portfolio(starting_bankroll)
        sim = FillSimulator(model, cfg, seed=cfg.get("synth_seed", 42))
        strat = HourlyArbStrategy(sim, cfg)
        for i, ep in enumerate(episodes):
            recs = strat.run(ep, pf.bankroll, episode_idx=i)
            for rec in recs:
                pf.apply(rec)
            if verbose and (i + 1) % 200 == 0:
                print(f"  [{model}] {i+1}/{len(episodes)}  bankroll=${pf.bankroll:,.2f}  trades={len(pf.records)}")
        m = compute_metrics(pf, len(episodes))
        m["portfolio"] = pf
        out[model] = m

    opt   = out["optimistic"]["total_pnl_dollars"]
    pess  = out["pessimistic"]["total_pnl_dollars"]
    real  = out["realistic"]["total_pnl_dollars"]
    real_abs = max(abs(real), 1.0)
    ratio = abs(opt - pess) / real_abs
    sharpe_real = out["realistic"].get("sharpe_annualized", 0.0)
    fragile = (ratio > cfg["model_risk_threshold"]
               or sharpe_real < cfg["min_acceptable_sharpe"])

    out["_summary"] = {
        "model_risk_ratio": ratio,
        "realistic_sharpe": sharpe_real,
        "fragile":          fragile,
        "verdict_text":     ("FRAGILE — fills dominate edge" if fragile
                             else "ROBUST — fill assumptions don't dominate"),
    }
    return out


def print_tri_model_summary(results: Dict):
    print("\n" + "=" * 90)
    print("TRI-MODEL VALIDATION")
    print("=" * 90)
    cols = ["PnL $", "PnL %", "trades", "win %", "fill %", "sharpe",
            "maxDD %", "profit fac", "slip $", "adv %"]
    header = f"  {'model':12s}  " + "  ".join(f"{c:>10s}" for c in cols)
    print(header)
    print("  " + "-" * (len(header) - 2))
    for model in ("optimistic", "realistic", "pessimistic"):
        m = results[model]
        cells = [
            f"{m['total_pnl_dollars']:>10,.2f}",
            f"{m['total_pnl_pct']:>10.2f}",
            f"{m.get('n_entries', 0):>10d}",
            f"{m.get('win_rate_pct', 0):>10.1f}",
            f"{m['fill_rate_pct']:>10.1f}",
            f"{m.get('sharpe_annualized', 0):>10.2f}",
            f"{m.get('max_drawdown_pct', 0):>10.1f}",
            f"{m.get('profit_factor', 0):>10.2f}",
            f"{m.get('avg_entry_slippage_dollars', 0):>10.4f}",
            f"{m.get('adverse_entry_rate_pct', 0):>10.1f}",
        ]
        print(f"  {model:12s}  " + "  ".join(cells))

    s = results["_summary"]
    print()
    print(f"  model_risk_ratio = {s['model_risk_ratio']:.2f}   "
          f"realistic_sharpe = {s['realistic_sharpe']:.2f}")
    print(f"  ➜ {s['verdict_text']}")

    # Per-strategy breakdown for realistic model
    pf = results["realistic"]["portfolio"]
    df = pf.trades_df()
    if not df.empty and "strategy" in df.columns:
        entered = df[df["entry_time"].notna()]
        if not entered.empty:
            print("\n── Per-Strategy Breakdown (realistic) ──")
            for strat_name, grp in entered.groupby("strategy"):
                n = len(grp)
                pnl = grp["pnl"].sum()
                wr = (grp["pnl"] > 0).mean() * 100
                avg = grp["pnl"].mean()
                print(f"  {strat_name:20s}  trades={n:>5d}  PnL=${pnl:>10,.2f}  "
                      f"win={wr:5.1f}%  avg=${avg:>8,.4f}")

## § 11  Regime Analysis

Slice trades by BTC realized vol regime (low / med / high tertile) and report
PnL & win rate separately. An edge that only exists in high-vol can still be
real — but you need to know it before deploying.

In [ ]:
def regime_analysis(records: List[TradeRecord],
                    episodes: List[HourlyEpisode]) -> Dict:
    if not records:
        return {"warning": "no records"}
    df = pd.DataFrame([asdict(r) for r in records])
    df = df[df["entry_time"].notna()].copy()
    if df.empty:
        return {"warning": "no entries"}
    rv_by_idx = {i: (ep.realized_vol or 0.0) for i, ep in enumerate(episodes)}
    df["rv"] = df["episode_idx"].map(rv_by_idx)
    if df["rv"].nunique() < 3:
        return {"warning": "insufficient vol regime variation"}
    qs = df["rv"].quantile([0.0, 1/3, 2/3, 1.0]).values
    df["regime"] = pd.cut(df["rv"], bins=qs, include_lowest=True,
                           labels=["low_vol", "med_vol", "high_vol"])
    out = {}
    for r in ["low_vol", "med_vol", "high_vol"]:
        sub = df[df["regime"] == r]
        out[r] = {
            "n": len(sub),
            "pnl_total": float(sub["pnl"].sum()),
            "avg_pnl":   float(sub["pnl"].mean()) if len(sub) else 0.0,
            "win_rate_pct": float((sub["pnl"] > 0).mean() * 100.0),
            "rv_range":  (float(sub["rv"].min()), float(sub["rv"].max())) if len(sub) else (0.0, 0.0),
        }
    return out


## § 12  Monte Carlo (synthetic)

`N` independent seeds × `H` hours of synthetic episodes. Reports the distribution
of final bankrolls and an equity-curve fan chart. Cheap stress test for the
strategy's path-sensitivity.

In [ ]:
def monte_carlo(n_seeds: int = 30, n_hours: int = 200,
                sigma_annual: float = 0.65, fill_model: str = "realistic",
                config: Dict = None) -> Dict:
    cfg = config or CONFIG
    finals: List[float] = []
    curves: List[np.ndarray] = []
    for seed in range(n_seeds):
        rng = np.random.default_rng(seed)
        s = 70_000.0 + float(rng.normal(0, 1000))
        t0 = datetime(2026, 4, 1, 0, 0, tzinfo=timezone.utc)
        eps = []
        for h in range(n_hours):
            ep = synthetic_episode(t0 + timedelta(hours=h), s0=s,
                                    sigma_annual=sigma_annual,
                                    seed=seed * 10_000 + h)
            eps.append(ep)
            s = float(ep.spot_prices.iloc[-1])
        sim = FillSimulator(fill_model, cfg, seed=seed)
        strat = HourlyArbStrategy(sim, cfg)
        pf = Portfolio(cfg["starting_bankroll"])
        for i, ep in enumerate(eps):
            for rec in strat.run(ep, pf.bankroll, episode_idx=i):
                pf.apply(rec)
        finals.append(pf.bankroll)
        eq = pf.equity_df()
        if not eq.empty:
            curves.append(eq["equity"].values)
    arr = np.array(finals)
    return {
        "n_seeds":      n_seeds,
        "n_hours":      n_hours,
        "fill_model":   fill_model,
        "mean_final":   float(arr.mean()),
        "median_final": float(np.median(arr)),
        "std_final":    float(arr.std()),
        "p5":           float(np.percentile(arr, 5)),
        "p95":          float(np.percentile(arr, 95)),
        "curves":       curves,
        "finals":       finals,
    }

## § 13  End-to-End Demo

Loads the first **N** valid episodes from the local DuckDB, runs all three fill
models, prints the comparison table and the regime breakdown, and saves
`output/tri_model_validation.png` and `output/trades_realistic.csv`.

Bump `N_DEMO` for a fuller backtest (every ~50 episodes ≈ 2 days of market data).

In [ ]:
# ── Run full tri-model validation on ALL available data ──────────────
demo_episodes: List[HourlyEpisode] = []
loader = None
if Path(CONFIG["duckdb_path"]).exists():
    loader = KalshiLocalLoader(CONFIG["duckdb_path"], CONFIG["bracket_width_dollars"])
    events = loader.list_events(limit=None)
    print(f"Found {len(events):,} hourly events with BTC coverage in DuckDB.")
    skipped = 0
    for et in events:
        ep = loader.load_event_full(et)
        if ep is None or ep.books.empty:
            skipped += 1
            continue
        demo_episodes.append(ep)
        if len(demo_episodes) % 200 == 0:
            print(f"  … loaded {len(demo_episodes)} episodes")
    print(f"  → loaded {len(demo_episodes)} episodes ({skipped} skipped).")
else:
    print(f"DuckDB not found at {CONFIG['duckdb_path']} — falling back to 200 synthetic episodes.")
    for h in range(200):
        demo_episodes.append(synthetic_episode(
            datetime(2026, 4, 1, 0, 0, tzinfo=timezone.utc) + timedelta(hours=h),
            seed=h,
        ))

print(f"\n→ Backtesting {len(demo_episodes)} episodes across 3 fill models…")
results = run_tri_model_validation(demo_episodes, CONFIG["starting_bankroll"], verbose=True)
print_tri_model_summary(results)

print("\n── REGIME ANALYSIS (realistic) ──")
regime = regime_analysis(results["realistic"]["portfolio"].records, demo_episodes)
for k, v in regime.items():
    print(f"  {k:10s} {v}")

if loader is not None:
    loader.close()

In [ ]:
# ── Plots & artefacts ────────────────────────────────────────────────
if not demo_episodes:
    print("No episodes — skipping plots.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    # 1) Equity curves overlaid
    ax = axes[0, 0]
    colors = {"optimistic": "#1f77b4", "realistic": "#2ca02c", "pessimistic": "#d62728"}
    for model in ("optimistic", "realistic", "pessimistic"):
        eq = results[model]["portfolio"].equity_df()
        if eq.empty:
            continue
        eq_pct = (eq["equity"] / CONFIG["starting_bankroll"] - 1.0) * 100.0
        ax.plot(range(len(eq_pct)), eq_pct.values,
                label=model, color=colors[model], linewidth=1.6)
    ax.axhline(0, color="black", alpha=0.3)
    ax.set_title("Equity curves (% of starting bankroll)")
    ax.set_xlabel("trade #"); ax.set_ylabel("PnL %")
    ax.legend(); ax.grid(alpha=0.3)

    # 2) Per-strategy PnL breakdown (realistic)
    ax = axes[0, 1]
    df = results["realistic"]["portfolio"].trades_df()
    df_e = df[df["entry_time"].notna()] if not df.empty else df
    if not df_e.empty and "strategy" in df_e.columns:
        strat_pnl = df_e.groupby("strategy")["pnl"].agg(["sum", "count"])
        strat_colors = {"parity": "#ff7f0e", "monotonicity": "#9467bd",
                        "convergence": "#17becf"}
        bars = ax.bar(strat_pnl.index,
                      strat_pnl["sum"],
                      color=[strat_colors.get(s, "#999") for s in strat_pnl.index])
        for bar, cnt in zip(bars, strat_pnl["count"]):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f"n={cnt}", ha="center", va="bottom", fontsize=9)
        ax.axhline(0, color="black", alpha=0.3)
        ax.set_title("PnL by strategy (realistic)")
        ax.set_ylabel("Total PnL ($)")
    else:
        ax.text(0.5, 0.5, "No trades", ha="center", va="center",
                transform=ax.transAxes)
        ax.set_title("PnL by strategy (realistic)")
    ax.grid(axis="y", alpha=0.3)

    # 3) Realistic-model PnL distribution
    ax = axes[1, 0]
    if not df_e.empty:
        ax.hist(df_e["pnl"], bins=30, color=colors["realistic"], alpha=0.75)
        ax.axvline(0, color="black", alpha=0.5, linestyle="--")
        ax.set_title("Realistic-model trade PnL ($)")
        ax.set_xlabel("$ PnL"); ax.grid(alpha=0.3)
    else:
        ax.text(0.5, 0.5, "No trades", ha="center", va="center",
                transform=ax.transAxes)
        ax.set_title("Realistic-model trade PnL ($)")

    # 4) Exit-reason breakdown (realistic)
    ax = axes[1, 1]
    if not df_e.empty:
        reason_counts = df_e["exit_reason"].value_counts()
        ax.barh(reason_counts.index, reason_counts.values,
                color=colors["realistic"], alpha=0.75)
        ax.set_title("Exit reasons (realistic)")
        ax.set_xlabel("count")
    else:
        ax.text(0.5, 0.5, "No trades", ha="center", va="center",
                transform=ax.transAxes)
        ax.set_title("Exit reasons (realistic)")
    ax.grid(axis="x", alpha=0.3)

    plt.tight_layout()
    out_png = OUTPUT_DIR / "tri_model_validation.png"
    plt.savefig(out_png, dpi=120, bbox_inches="tight")
    plt.show()

    # Save the realistic-model trade ledger
    if not df_e.empty:
        out_csv = OUTPUT_DIR / "trades_realistic.csv"
        df_e.to_csv(out_csv, index=False)
        print(f"\nSaved {out_png}")
        print(f"Saved {out_csv}")

    # Final verdict
    s = results["_summary"]
    print("\n" + "=" * 70)
    print(f"  VERDICT: {s['verdict_text']}")
    print(f"  realistic Sharpe = {s['realistic_sharpe']:.2f}    "
          f"model risk ratio = {s['model_risk_ratio']:.2f}")
    print("=" * 70)